In [13]:
# Dataset manipulation
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
import category_encoders as ce
from optbinning import OptimalBinning
import utils_pipeline_dhm as up

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

# Metrics
from sklearn.metrics import (recall_score, 
                             classification_report,
                             roc_auc_score,
                             average_precision_score
                            )

# Others
import importlib

In [14]:
# read data
data_path = '../data/diabetes_prediction_dataset.csv'
data = pd.read_csv(data_path, sep=',')
print('Filas del dataset en bruto:', data.shape[0])
# Filter data
data = data[data['bmi'] < 65]
data = data[data['gender']!='Other']
print('Filas del dataset después de filtrar:', data.shape[0])
# check data
data.head()

Filas del dataset en bruto: 100000
Filas del dataset después de filtrar: 99935


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [15]:
# Features and target
X = data.drop(columns=["diabetes"])
y = data["diabetes"]   # binary: 0/1

# Split first (important to avoid leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [16]:
# Declare columns
binary_cols = ['hypertension','heart_disease']
woe_cat_cols = ['gender','smoking_history','hypertension','heart_disease']

# Numerical columns to bin and their max number of bins
binning_config = {
    "blood_glucose_level": 4,
    'HbA1c_level': 5,
    "age": 8,
    "bmi": 8
}

# Transformations
binary_to_cat = up.TypeCaster(
    columns=binary_cols,
    dtype=str
)

woe_transformer = Pipeline([
    ("woe", ce.WOEEncoder()),
    ("invert", FunctionTransformer(lambda X: -X,feature_names_out="one-to-one"))
])

# Preprocessor
preprocessor = ColumnTransformer([
    ("woe", woe_transformer, woe_cat_cols),
    ("bin_woe",up.MultiOptimalBinningWOE(binning_config),list(binning_config.keys()))
    ],
remainder="passthrough",
verbose_feature_names_out=True
)

# Pipeline
log_pipe_line = Pipeline([
    ("type_cast", binary_to_cat),
    ("preprocessor", preprocessor),
    ("model", LogisticRegression())
])


In [ ]:
scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

param_grid = {
    "model__C": [0.01, 0.1]
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

log_grid = GridSearchCV(
    log_pipe_line,
    param_grid=param_grid,
    cv=cv,
    scoring=scoring,
    refit="pr_auc",
    n_jobs=-1
)

log_grid.fit(X_train, y_train)

cv_results = pd.DataFrame(log_grid.cv_results_)
cv_results = cv_results[["param_model__C", "mean_test_roc_auc", "mean_test_pr_auc",'std_test_roc_auc', 'std_test_pr_auc']]
print(cv_results,'\n')

# best model
best_params = log_grid.best_params_
best_score = log_grid.best_score_

print("Stats for best parameters:")
print("Best Parameters:", best_params)
print("Mean PR-AUC Score:", best_score,'\n')

# Get the best model and evaluate on both Train and Test sets
best_model = log_grid.best_estimator_

# Print Train metrics for the best model
print("Train Metrics for the Best Model:")
train_preds = best_model.predict_proba(X_train)[:,1]
print('Train PR-AUC',average_precision_score(y_train, train_preds))
print('Train ROC-AUC',roc_auc_score(y_train, train_preds),'\n')

# print Test metrics for the best model
print("Test Metrics for the Best Model:")
test_preds = best_model.predict_proba(X_test)[:,1]
print('Test PR-AUC',average_precision_score(y_test, test_preds))
print('Test AUC',roc_auc_score(y_test, test_preds))

   param_model__C  mean_test_roc_auc  mean_test_pr_auc  std_test_roc_auc  \
0            0.01           0.936459          0.679369          0.002573   
1            0.10           0.936499          0.679768          0.002500   

   std_test_pr_auc  
0         0.005083  
1         0.004789   

Best Parameters: {'model__C': 0.1}
Best PR-AUC Score, during CV: 0.6797675986277764 

Train Metrics for the Best Model:
Train PR-AUC 0.6820129031128195
Train ROC-AUC 0.9374008900503993 

Test Metrics for the Best Model:
Test PR-AUC 0.6810205538477876
Test AUC 0.940693479278552


In [ ]:
# log_pipe_line.fit(X_train, y_train)
# 
# feature_names = log_pipe_line.named_steps["preprocessor"].get_feature_names_out()
# 
# df_processed = log_pipe_line.transform(X_train)
# df_processed = pd.DataFrame(df_processed,columns=feature_names)
# df_processed.head()

#### Check the sense of the woe, if this is the same in optimal binning that in the woe_encoder.

Once that is ok we can adjust the logistic regresion.

In [6]:
# Get WOE mapping
#woe_encoder = log_pipe_line.named_steps["preprocessor"].named_transformers_["woe"]
importlib.reload(up)  # reload utils to reflect recent changes
woe_encoder = (
    log_pipe_line
    .named_steps["preprocessor"]
    .named_transformers_["woe"]
    .named_steps["woe"]
)

woe_map = up.get_woe_mapping(woe_encoder)
woe_map

,feature,cat,ord_enc,woe_enc
0,gender,Male,1,-0.149763
1,gender,Female,2,0.118977
3,smoking_history,current,1,-0.223560
4,smoking_history,No Info,2,0.774097
5,smoking_history,never,3,-0.118364
6,smoking_history,not current,4,-0.254523
7,smoking_history,former,5,-0.784648
8,smoking_history,ever,6,-0.375887
10,hypertension,0,1,0.225959
11,hypertension,1,2,-1.438288


In [8]:
# get binning and WOE mapping
bin_woe_encoder = log_pipe_line.named_steps["preprocessor"].named_transformers_["bin_woe"]
bin_woe_map = up.get_bin_woe_mapping(bin_woe_encoder)
bin_woe_map
#bin_woe_map[bin_woe_map['feature']=='blood_glucose_level']

,feature,Bin,WoE,IV
0,blood_glucose_level,"(-inf, 128.00)",1.612593,0.497041
1,blood_glucose_level,"[128.00, 159.50)",0.199551,0.016705
2,blood_glucose_level,"[159.50, 180.00)",-0.075264,0.000451
3,blood_glucose_level,"[180.00, inf)",-1.809115,0.698225
4,blood_glucose_level,Special,0.000000,0.000000
5,blood_glucose_level,Missing,0.000000,0.000000
6,HbA1c_level,"(-inf, 5.75)",1.809661,0.751757
7,HbA1c_level,"[5.75, 5.90)",0.121046,0.001170
8,HbA1c_level,"[5.90, 6.55)",0.094640,0.002844
9,HbA1c_level,"[6.55, inf)",-1.842499,0.838581
